# Boussinesq CNO — actual saved-model visualization

**No training or synthetic images.** This notebook loads the CNO N=200 best-validation checkpoint and the original AWFNO and FNO checkpoints when found, using the same **70 held-out test trajectories**. It compares **true temperature c_grid vs CNO vs AWFNO vs FNO vs absolute CNO error** on the same annular domain and fixed color scale, at 5 times for best / median / worst CNO cases (ranked by final-time physical L2).

Additional measurements: angular Fourier high-frequency power retention, high-gradient-region relative L2, standard deviation and angular spectra. Saved to MyDrive/anus/Boussinesq_JCP/native_reference_2d/CNO_visual_audit/.

**Note:** Historical FNO here is the paper's direct full-trajectory FNO, not the new autoregressive F-FNO. If an AWFNO/FNO checkpoint is unavailable, its panels are labelled unavailable.

In [ ]:
#@title 1. Google Drive, correct model implementations, GPU
from google.colab import drive
from pathlib import Path
import os,sys,time,urllib.request,hashlib,json,subprocess
import numpy as np,torch,matplotlib.pyplot as plt,pandas as pd
from IPython.display import display,Image
drive.mount('/content/gdrive')
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
DATASET=Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
PROJECT=Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
NATIVE=PROJECT/'native_reference_2d'
OUT=NATIVE/'CNO_visual_audit'
N_TRAIN=200 #@param {type:'integer'}
SEED=1234 #@param {type:'integer'}
assert N_TRAIN in (25,50,100,200)
assert DATASET.is_file() and NATIVE.is_dir(), 'Original dataset or native baseline results missing.'
OUT.mkdir(parents=True,exist_ok=True)
os.environ.update(DATA_PATH=str(DATASET),OUTPUT_ROOT=str(PROJECT),NATIVE_OUTPUT=str(NATIVE),
                  SPLIT_MODE='contiguous',TIME_SUBSAMPLE='1',N_VAL='30',N_TEST='70')
subprocess.run([sys.executable,'-m','pip','install','-q','neuraloperator==2.0.0'],check=True)
GITHUB='https://raw.githubusercontent.com/dimigourz/AWFNO/main/Boussinesq2D/'
for name in ['annulus2d_jcp_experiments.py','reference_architectures_2d.py','native_method_baselines_2d.py']:
    request=urllib.request.Request(GITHUB+name+'?v='+str(time.time_ns()),headers={'Cache-Control':'no-cache'})
    with urllib.request.urlopen(request,timeout=120) as rr:data=rr.read()
    (Path('/content')/name).write_bytes(data)
    print('[CODE]',name,'SHA256=',hashlib.sha256(data).hexdigest())
if '/content' not in sys.path:sys.path.insert(0,'/content')
import annulus2d_jcp_experiments as base
import native_method_baselines_2d as native
from reference_architectures_2d import make_native_model
print('[GPU]',torch.cuda.get_device_name(0),'[OUTPUT]',OUT)

In [ ]:
#@title 2. Load genuine CNO best-validation checkpoint; verify reported test error
U,r,theta,tgrid,sample_ids=base.load_annulus_dataset(DATASET)
U,r,theta=base.downsample_polar_grid(U,r,theta,base.TARGET_NR,base.TARGET_NTHETA)
base.set_geometry_from_arrays(U.shape,r,theta,tgrid)
pool,train_ids,val,val_ids,test,test_ids=base.make_train_val_test_splits(U)
assert (pool.shape[0],val.shape[0],test.shape[0])==(400,30,70)
assert test.shape[1:]==(64,128,101)
norm=native.StateNormalizer(pool[:N_TRAIN])
runs=sorted((NATIVE/'runs'/'CNO').glob(f'N{N_TRAIN}_seed{SEED}_*/result.json'))
runs=[p for p in runs if (p.parent/'training_checkpoint.pt').is_file() and
      json.loads(p.read_text()).get('model')=='CNO']
assert runs,'No finished CNO model of the specified size/seed is present.'
if len(runs)>1:
    newest=[p for p in runs if json.loads(p.read_text()).get('training_protocol_version')==native.VERSION]
    if len(newest)==1:runs=newest
    else:raise RuntimeError('More than one completed CNO run; select a checkpoint explicitly.')
saved=json.loads(runs[0].read_text())
checkpoint=runs[0].parent/'training_checkpoint.pt'
payload=torch.load(checkpoint,map_location='cpu',weights_only=False)
assert isinstance(payload.get('best_model'),dict), 'Best checkpoint not found.'
CNO=make_native_model('CNO',r,theta).to(base.device)
CNO.load_state_dict(payload['best_model'],strict=True)
CNO.eval()
print('[MODEL]',checkpoint,'best_epoch',payload.get('best_epoch'))
@torch.inference_mode()
def cno_at(indices,tidx):
    output=[]
    tvalue=float((tgrid[tidx]-tgrid[0])/(tgrid[-1]-tgrid[0]))
    for k in range(0,len(indices),8):
        ids=indices[k:k+8]
        if tidx==0:output.append(test[ids,:,:,0]);continue
        x=torch.from_numpy(test[ids,:,:,0]).to(base.device)
        ts=torch.full((len(ids),),tvalue,device=base.device)
        y=norm.decode(CNO(norm.encode(x),ts))
        assert torch.isfinite(y).all()
        output.append(y.detach().float().cpu().numpy())
    return np.concatenate(output)
pred_final=cno_at(list(range(70)),100)
final_scores=base.l2_per_timestep_np(pred_final[...,None],test[...,-1:])[:,0]
print('[RECOMPUTED FINAL L2]',float(final_scores.mean()),' [SAVED]',saved['final_l2_mean'])
assert abs(final_scores.mean()-saved['final_l2_mean'])<max(1.e-4,0.002*saved['final_l2_mean']), (
       'Data/model mismatch: saved and recomputed final error differ.')
sortidx=np.argsort(final_scores)
cases={'best':int(sortidx[0]),'median':int(sortidx[len(sortidx)//2]),'worst':int(sortidx[-1])}
for label,i in cases.items():
    print(f'[{label}] test index={i}, original trajectory ID={test_ids[i]}, CNO final L2={final_scores[i]:.6f}')

In [ ]:
#@title 3. OPTIONAL: load original AWFNO and direct FNO without retraining
from itertools import chain
cache_roots=sorted(PROJECT.glob('cache_annulus2d_awfno_sample_sweep_*'))
def find_original(label):
    paths=list(chain.from_iterable((d/'models').glob(f'N{N_TRAIN}_{label}_*.pt') for d in cache_roots))
    if len(paths)==1:return paths[0]
    print('[OPTIONAL]',label,':',len(paths),'checkpoints; skipping to avoid wrong model.')
    return None
AWFNO=None;FNO=None
p_aw=find_original('AWFNO')
p_fn=find_original('FNO_neuralop_full')
if p_aw is not None:
    try:
        starts=base.make_window_start_indices(base.Nt,base.WINDOW_K,base.WINDOW_STRIDE)
        inputs,targets=base.build_window_tensors(pool[:N_TRAIN],starts)
        n_aw_in=base.GaussianNormalizer(torch.from_numpy(inputs)).to(base.device)
        n_aw_out=base.GaussianNormalizer(torch.from_numpy(targets)).to(base.device)
        AWFNO=base.AWFNO2d(base.WINDOW_K,r,theta).to(base.device)
        base.load_model_checkpoint(p_aw,AWFNO);AWFNO.eval()
        del inputs,targets
        print('[AWFNO] checkpoint loaded:',p_aw)
    except Exception as exc:print('[AWFNO skipped]',repr(exc));AWFNO=None
if p_fn is not None:
    try:
        n_fn_in=base.GaussianNormalizer(torch.from_numpy(pool[:N_TRAIN,...,0])).to(base.device)
        n_fn_out=base.GaussianNormalizer(torch.from_numpy(pool[:N_TRAIN,...,1:])).to(base.device)
        FNO=base.NeuralOperatorFNOFull2d(base.Nt,r,theta).to(base.device)
        base.load_model_checkpoint(p_fn,FNO);FNO.eval()
        print('[FNO] checkpoint loaded:',p_fn)
    except Exception as exc:print('[FNO skipped]',repr(exc));FNO=None
@torch.inference_mode()
def aw_rollout(index):
    if AWFNO is None:return None
    pred=np.empty_like(test[index])
    pred[...,0]=test[index,...,0]
    state=torch.as_tensor(test[index,...,0][None],device=base.device)
    cursor=0
    while cursor<100:
        block=n_aw_out.decode(AWFNO(n_aw_in.encode(state)))
        k=min(base.WINDOW_K,100-cursor)
        pred[...,cursor+1:cursor+k+1]=block[0,...,:k].detach().cpu().numpy()
        state=block[...,k-1].detach()
        cursor+=k
    return pred
@torch.inference_mode()
def fno_rollout(index):
    if FNO is None:return None
    x=torch.from_numpy(test[index,...,0][None]).to(base.device)
    future=n_fn_out.decode(FNO(n_fn_in.encode(x))).detach().cpu().numpy()[0]
    return np.concatenate([test[index,...,0,None],future],axis=-1)

In [ ]:
#@title 4. DISPLAY: true temperature versus CNO/AWFNO/FNO, and CNO absolute error
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
FRAMES=[20,40,60,80,100]
th_closed=np.concatenate([theta,[theta[0]+2*np.pi]])
R,TH=np.meshgrid(r,th_closed,indexing='ij')
X,Y=R*np.cos(TH),R*np.sin(TH)
def draw(ax,v,lo,hi,cmap):
    closed=np.concatenate([v,v[:,:1]],axis=1)
    ax.pcolormesh(X,Y,closed,cmap=cmap,vmin=lo,vmax=hi,shading='auto',rasterized=True)
    ax.set_aspect('equal');ax.axis('off')
summary=[]
for label,idx in cases.items():
    truth=test[idx]
    cno={t:cno_at([idx],t)[0] for t in FRAMES}
    ap=aw_rollout(idx)
    fp=fno_rollout(idx)
    observed=[truth[...,t] for t in FRAMES]+list(cno.values())
    if ap is not None:observed.extend(ap[...,t] for t in FRAMES)
    if fp is not None:observed.extend(fp[...,t] for t in FRAMES)
    lo=min(float(x.min()) for x in observed);hi=max(float(x.max()) for x in observed)
    errmax=max(1e-8,max(float(np.max(np.abs(cno[t]-truth[...,t]))) for t in FRAMES))
    fig,axes=plt.subplots(5,5,figsize=(18,17),constrained_layout=True)
    headings=['Ground truth','CNO prediction','AWFNO' if ap is not None else 'AWFNO missing',
              'Direct FNO' if fp is not None else 'FNO missing','|CNO - truth|']
    for row,t in enumerate(FRAMES):
        correct=truth[...,t];estimate=cno[t]
        images=[correct,estimate,None if ap is None else ap[...,t],
                None if fp is None else fp[...,t],np.abs(estimate-correct)]
        for col,img in enumerate(images):
            ax=axes[row,col]
            if img is None:ax.text(.5,.5,'Checkpoint unavailable',ha='center',va='center',transform=ax.transAxes);ax.axis('off')
            else:draw(ax,img,0 if col==4 else lo,errmax if col==4 else hi,'magma' if col==4 else 'viridis')
            if row==0:ax.set_title(headings[col],fontsize=12)
            if col==0:ax.text(-.10,.5,f't index {t}',ha='center',va='center',
                               rotation=90,transform=ax.transAxes)
        true_modes=np.fft.rfft(correct,axis=1)[:,16:]
        pred_modes=np.fft.rfft(estimate,axis=1)[:,16:]
        high_ratio=float(np.sum(np.abs(pred_modes)**2)/max(np.sum(np.abs(true_modes)**2),1e-20))
        dr=np.gradient(correct,r,axis=0)
        dtheta=(np.roll(correct,-1,axis=1)-np.roll(correct,1,axis=1))/(2*(2*np.pi/128)*r[:,None])
        edge=(dr**2+dtheta**2)>np.quantile(dr**2+dtheta**2,.90)
        sharp_error=float(np.sqrt(np.sum((estimate-correct)**2*edge*r[:,None])/
                                  max(np.sum(correct**2*edge*r[:,None]),1e-20)))
        summary.append(dict(case=label,test_idx=idx,time_idx=t,angular_high_mode_energy_ratio=high_ratio,
                            sharp_region_relative_L2=sharp_error,truth_std=float(correct.std()),
                            CNO_std=float(estimate.std())))
    fig.suptitle(f'Boussinesq TEST: {label} CNO case #{idx} | N={N_TRAIN} | common field colour scale')
    fig.colorbar(ScalarMappable(norm=Normalize(lo,hi),cmap='viridis'),ax=axes[:,:4].ravel().tolist(),
                 location='bottom',fraction=.02,pad=.02,label='Temperature c: common scale')
    fig.colorbar(ScalarMappable(norm=Normalize(0,errmax),cmap='magma'),ax=axes[:,4].ravel().tolist(),
                 location='bottom',fraction=.02,pad=.02,label='CNO absolute error')
    output=OUT/f'CNO_ground_truth_AWFNO_FNO_N{N_TRAIN}_{label}.png'
    fig.savefig(output,dpi=145)
    plt.show();plt.close(fig)
    print('[SAVED]',output)
stats=pd.DataFrame(summary)
stats.to_csv(OUT/f'CNO_structural_diagnostics_N{N_TRAIN}.csv',index=False)
display(stats.round(5))
print('Angular high-mode ratio below 1 indicates less high-frequency energy than truth; compare with structure/error maps.')

In [ ]:
#@title 5. Median-case angular Fourier power: is CNO oversmoothing?
idx=cases['median']
truth=test[idx,...,100];pred=cno_at([idx],100)[0]
Etrue=(np.abs(np.fft.rfft(truth,axis=1))**2).sum(axis=0)
Epred=(np.abs(np.fft.rfft(pred,axis=1))**2).sum(axis=0)
fig,ax=plt.subplots(figsize=(9,5),constrained_layout=True)
ax.semilogy(Etrue+1e-18,label='Truth')
ax.semilogy(Epred+1e-18,label='CNO')
ax.axvline(16,color='0.5',linestyle='--',label='high-mode cutoff')
ax.set_xlabel('Azimuthal Fourier mode');ax.set_ylabel('Total angular spectral energy')
ax.set_title('CNO versus truth, median final-time test case');ax.legend();ax.grid(alpha=.25)
output=OUT/f'CNO_high_frequency_spectrum_N{N_TRAIN}_median.png'
fig.savefig(output,dpi=160)
plt.show();plt.close(fig)
print('[SAVED]',output)